# 线性模型（下）Logistic 回归与 Softmax 回归

在线性模型的基础上加非线性激活，做二分类（Logistic）和多分类（Softmax）。

## Logistic 回归（二分类）

模型：$P(y=1\mid x) = \sigma(w^\top x + b)$。损失用二元交叉熵。

### 生成数据

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)

def make_two_clusters(n=200):
    pos = torch.randn(n, 2) + torch.tensor([2.0, 2.0])
    neg = torch.randn(n, 2) + torch.tensor([-2.0, -2.0])
    X = torch.cat([pos, neg], dim=0)
    y = torch.cat([torch.ones(n), torch.zeros(n)]).unsqueeze(1)
    perm = torch.randperm(len(X))
    return X[perm], y[perm]

X, y = make_two_clusters(200)
plt.scatter(X[y.squeeze() == 1, 0], X[y.squeeze() == 1, 1], c='r', s=8, label='1')
plt.scatter(X[y.squeeze() == 0, 0], X[y.squeeze() == 0, 1], c='b', s=8, label='0')
plt.legend(); plt.show()

### 训练

用 `BCEWithLogitsLoss`：它把 sigmoid 和 BCE 融合，数值上比先 sigmoid 再 BCE 更稳定。

In [ ]:
model = nn.Linear(2, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(200):
    logits = model(X)
    loss = loss_fn(logits, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

with torch.no_grad():
    pred = (torch.sigmoid(model(X)) >= 0.5).float()
    acc = (pred == y).float().mean().item()
print(f'final loss {loss.item():.4f}  accuracy {acc:.4f}')

### 决策边界

In [ ]:
xx, yy = torch.meshgrid(
    torch.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
    torch.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200),
    indexing='xy',
)
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
with torch.no_grad():
    prob = torch.sigmoid(model(grid)).reshape(xx.shape)

plt.contourf(xx.numpy(), yy.numpy(), prob.numpy(), levels=20, alpha=0.4, cmap='coolwarm')
plt.scatter(X[y.squeeze() == 1, 0], X[y.squeeze() == 1, 1], c='r', s=8)
plt.scatter(X[y.squeeze() == 0, 0], X[y.squeeze() == 0, 1], c='b', s=8)
plt.title('Logistic regression decision boundary'); plt.show()

## Softmax 回归（多分类）

模型：$P(y=k\mid x) = \dfrac{\exp(w_k^\top x + b_k)}{\sum_j \exp(w_j^\top x + b_j)}$。

PyTorch 里直接用 `nn.CrossEntropyLoss`，它接受原始 logits 与整数标签，内部已经做了 log-softmax + NLL，**模型最后一层不要再加 softmax**。

### 生成数据

In [ ]:
torch.manual_seed(0)

def make_three_clusters(n=150):
    c0 = torch.randn(n, 2) + torch.tensor([ 0.0,  3.0])
    c1 = torch.randn(n, 2) + torch.tensor([-3.0, -2.0])
    c2 = torch.randn(n, 2) + torch.tensor([ 3.0, -2.0])
    X = torch.cat([c0, c1, c2], dim=0)
    y = torch.cat([torch.full((n,), i, dtype=torch.long) for i in range(3)])
    perm = torch.randperm(len(X))
    return X[perm], y[perm]

X, y = make_three_clusters(150)
colors = ['r', 'g', 'b']
for k in range(3):
    plt.scatter(X[y == k, 0], X[y == k, 1], c=colors[k], s=8, label=f'class {k}')
plt.legend(); plt.show()

### 训练

In [ ]:
model = nn.Linear(2, 3)         # 3 类，输出 3 维 logits
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(300):
    logits = model(X)
    loss = loss_fn(logits, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

with torch.no_grad():
    pred = model(X).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'final loss {loss.item():.4f}  accuracy {acc:.4f}')

### 决策边界

In [ ]:
xx, yy = torch.meshgrid(
    torch.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
    torch.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200),
    indexing='xy',
)
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
with torch.no_grad():
    region = model(grid).argmax(dim=1).reshape(xx.shape)

plt.contourf(xx.numpy(), yy.numpy(), region.numpy(), levels=[-0.5, 0.5, 1.5, 2.5],
             colors=['#ffaaaa', '#aaffaa', '#aaaaff'], alpha=0.6)
for k in range(3):
    plt.scatter(X[y == k, 0], X[y == k, 1], c=colors[k], s=8, label=f'class {k}')
plt.legend(); plt.title('Softmax regression decision regions'); plt.show()